# Pharmaceutical R&D Document Research & Critique
Agentic deep research and critique of pharmaceutical documents using Dataiku LLM Mesh + SerpAPI.

In [ ]:
# Run once to install dependencies (comment out after first run)
# %pip install -r ../requirements.txt

In [ ]:
import sys
import os

sys.path.insert(0, "..")

from IPython.display import display, Markdown, clear_output

from src import ResearchCritiqueSystem, Config
from src.llm.mesh_client import MeshClient

In [ ]:
# ── Discover available LLMs and load SerpAPI key from project variables ────
available_llms = MeshClient.list_llm_ids()

if not available_llms:
    available_llms = [
        "openai:gpt-4o",
        "openai:gpt-4o-mini",
        "anthropic:claude-sonnet-4-6",
        "anthropic:claude-haiku-4-5",
    ]
    print("Could not query Dataiku Mesh — using fallback LLM list.")
else:
    print("Available LLMs:")
    for llm in available_llms:
        print(f"  {llm}")

_serp_key_from_vars = ""
try:
    import dataiku
    _proj_vars = dataiku.api_client().get_project(dataiku.default_project_key()).get_variables()
    _serp_key_from_vars = _proj_vars.get("standard", {}).get("serp_api_key", "")
    if _serp_key_from_vars:
        print("\nserp_api_key loaded from project variables.")

In [ ]:
# ═══════════════════════════════════════════════════════
# CONFIGURE HERE — edit these variables then run this cell
# ═══════════════════════════════════════════════════════

# Paste one of the LLM IDs printed above
LLM_ID = available_llms[0] if available_llms else "openai:gpt-4o"

# SerpAPI key — pre-filled from project variables, override if needed
SERP_API_KEY = _serp_key_from_vars or "YOUR_SERP_API_KEY"

# Describe the analysis you want
ANALYSIS_DESCRIPTION = (
    "Evaluate this document for scientific rigor, regulatory alignment "
    "with FDA/EMA guidelines, and identify evidence gaps."
)

# Agent tuning
MAX_AGENT_STEPS = 12
MIN_WEB_SEARCHES = 3

# ── Validation ─────────────────────────────────────────
_ok = True
if not SERP_API_KEY or SERP_API_KEY == "YOUR_SERP_API_KEY":
    print("⚠ Set SERP_API_KEY above.")
    _ok = False
if not ANALYSIS_DESCRIPTION:
    print("⚠ Set ANALYSIS_DESCRIPTION above.")
    _ok = False
if _ok:
    print(f"✓ LLM:  {LLM_ID}")
    print("✓ Ready — upload your document above if not done, then run the next cell.")

In [ ]:
# ── Run analysis ───────────────────────────────────────────────────────────
import tempfile, pathlib

# Resolve uploaded file — handles ipywidgets 7.x (dict) and 8.x (tuple of dicts)
def _save_upload(upload_widget):
    val = upload_widget.value
    if not val:
        raise ValueError("No file uploaded. Use the upload widget above first.")
    # ipywidgets 8.x: val is a tuple of dicts with keys 'name', 'content', etc.
    # ipywidgets 7.x: val is a dict keyed by filename
    if isinstance(val, dict):
        fname = list(val.keys())[0]
        content = val[fname]["content"]
    else:
        file_info = val[0]
        fname = file_info["name"]
        content = bytes(file_info["content"])
    suffix = pathlib.Path(fname).suffix or ".pdf"
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix, prefix=fname.replace(suffix, "") + "_")
    tmp.write(content if isinstance(content, bytes) else bytes(content))
    tmp.close()
    return tmp.name, fname

DOCUMENT_PATH, _orig_name = _save_upload(doc_upload)
print(f"Document: {_orig_name}  →  {DOCUMENT_PATH}")

config = Config(
    llm_id=LLM_ID,
    serp_api_key=SERP_API_KEY,
    max_iterations=MAX_AGENT_STEPS,
    min_searches_required=MIN_WEB_SEARCHES,
)

system = ResearchCritiqueSystem(config)
accumulated = []

for chunk in system.run_stream(file_path=DOCUMENT_PATH, description=ANALYSIS_DESCRIPTION):
    accumulated.append(chunk)
    clear_output(wait=True)
    display(Markdown("".join(accumulated)))

In [ ]:
# ── Run analysis ───────────────────────────────────────────────────────────

config = Config(
    llm_id=LLM_ID,
    serp_api_key=SERP_API_KEY,
    max_iterations=MAX_AGENT_STEPS,
    min_searches_required=MIN_WEB_SEARCHES,
)

system = ResearchCritiqueSystem(config)
accumulated = []

for chunk in system.run_stream(file_path=DOCUMENT_PATH, description=ANALYSIS_DESCRIPTION):
    accumulated.append(chunk)
    clear_output(wait=True)
    display(Markdown("".join(accumulated)))

In [ ]:
# ── Optional: save report to file ─────────────────────────────────────────
# Run this cell after the analysis completes.

# output_path = DOCUMENT_PATH.rsplit(".", 1)[0] + "_critique.md"
# with open(output_path, "w") as f:
#     f.write(system._last_report.raw_markdown)
# print(f"Report saved to: {output_path}")